# 多分类问题

In [1]:
import torch
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim

In [3]:
batch_size  = 64
transform = transforms.Compose([ # Compose类似管道操作符，将列表中的操作组合
    transforms.ToTensor(), # 将每张图片的28*28像素值转化为1*28*28的张量
    # 1表示通道数，1*28*28个灰度值会转化为0~1的数
    transforms.Normalize((0.1307,), (0.3081,)),# MNIST像素值的均值和方差
    # 若为三通道，每个括号将会有三个值
])

train_dataset = datasets.MNIST(
    '../mnist', 
    train=True, 
    download=True, 
    transform=transform) # 在这里完成图像输入的转化
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True)

# 测试集的数据加载
test_dataset = datasets.MNIST(
    '../mnist', 
    train=False, 
    download=True, 
    transform=transform)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=True)

In [4]:
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.l1 = torch.nn.Linear(784, 512)
        self.l2 = torch.nn.Linear(512, 256)
        self.l3 = torch.nn.Linear(256, 128)
        self.l4 = torch.nn.Linear(128, 64)
        self.l5 = torch.nn.Linear(64, 10) # 最终对应十个类别

    def forward(self, x):
        x = x.view(-1,784) # 输入的x是[batch_size, 1, 28, 28]，
        # 将x灰度值矩阵拉成[batch_size, 784]的矩阵，-1表示自动计算batch_size
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = F.relu(self.l3(x))
        x = F.relu(self.l4(x))
        return self.l5(x)

model = Net()

In [5]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.5)
# 优化器引入冲量，能使梯度下降更稳定

In [6]:
# 这里采用将每一轮训练定义为一个函数
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, labels = data
        # input是64个1*28*28的张量
        # label是64个数字
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 300 == 299: # 每300个batch打印一次
            print('[%d, %5d] loss: %.3f' % (epoch + 1, batch_idx + 1, running_loss / 300))
            # 整数、五位整数、三位小数；%表示将括号内的数值填入
            running_loss = 0.0

In [7]:
def test():
    correct = 0
    total = 0
    with torch.no_grad(): # 因为不需要算梯度了
        for data in test_loader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1)
            # outputs.data是n*10的矩阵，每一行代表一个样本的10个分类概率
            # dim=1表示按行（第一维）找最大值，返回最大值_和最大值对应的索引predicted
            total += labels.size(0) # 将batch数加入total
            correct += (predicted == labels).sum().item()
        print("测试集的准确率：{}".format(correct / total))

In [8]:
%%time
if __name__=="__main__":
    for epoch in range(10):
        train(epoch)
        test()

[1,   300] loss: 2.170
[1,   600] loss: 0.848
[1,   900] loss: 0.420
测试集的准确率：0.9
[2,   300] loss: 0.312
[2,   600] loss: 0.283
[2,   900] loss: 0.234
测试集的准确率：0.9426
[3,   300] loss: 0.195
[3,   600] loss: 0.177
[3,   900] loss: 0.159
测试集的准确率：0.9515
[4,   300] loss: 0.133
[4,   600] loss: 0.130
[4,   900] loss: 0.121
测试集的准确率：0.9645
[5,   300] loss: 0.100
[5,   600] loss: 0.095
[5,   900] loss: 0.096
测试集的准确率：0.9689
[6,   300] loss: 0.076
[6,   600] loss: 0.081
[6,   900] loss: 0.080
测试集的准确率：0.9726
[7,   300] loss: 0.057
[7,   600] loss: 0.060
[7,   900] loss: 0.071
测试集的准确率：0.9728
[8,   300] loss: 0.050
[8,   600] loss: 0.053
[8,   900] loss: 0.051
测试集的准确率：0.9748
[9,   300] loss: 0.040
[9,   600] loss: 0.042
[9,   900] loss: 0.042
测试集的准确率：0.9724
[10,   300] loss: 0.030
[10,   600] loss: 0.033
[10,   900] loss: 0.035
测试集的准确率：0.9773
CPU times: total: 2min 4s
Wall time: 1min 12s
